## 4. División del conjunto de datos y transformaciones

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.utils import shuffle

# Paths
preprocessed_input_N = Path("../data/processed/preprocessed_data_N.parquet")
refined_input_N = Path("../data/processed/refined_data_N.parquet")

X_train_out = Path("../data/modeling/supervised/X_train.parquet")
X_test_out = Path("../data/modeling/supervised/X_test.parquet")
y_train_out = Path("../data/modeling/supervised/y_train.parquet")
y_test_out = Path("../data/modeling/supervised/y_test.parquet")

X_train_N_out = Path("../data/modeling/unsupervised/X_train_N.parquet")
X_test_N_out = Path("../data/modeling/unsupervised/X_test_N.parquet")
y_test_N_out = Path("../data/modeling/unsupervised/y_test_N.parquet")

# Add src/ to Python path
sys.path.append(str(Path("../src").resolve()))
from data.var_type import split_symbolic_continuous
from data.cat_encoding import CustomBenignFirstEncoder

#### 1) Revisión general

In [2]:
dn_log = pd.read_parquet(refined_input_N)
dn = pd.read_parquet(preprocessed_input_N)

Primero de todo encuentro las variables numéricas y las categóricas/simbólicas para ambos dataset

In [3]:
symbolic, continuous = split_symbolic_continuous(dn)
symbolicN, continuousN = split_symbolic_continuous(dn_log)

# For dn are the same because the only diff is we add another numeric variables so, symbolic = symbolicN
for col in symbolic:
    print(col, sorted(dn[col].unique())[:10])

Protocol [np.int64(0), np.int64(6), np.int64(17)]
Fwd PSH Flags [np.uint32(0), np.uint32(1)]
Fwd URG Flags [np.uint32(0), np.uint32(1)]
FIN Flag Cnt [np.uint32(0), np.uint32(1)]
SYN Flag Cnt [np.uint32(0), np.uint32(1)]
RST Flag Cnt [np.uint32(0), np.uint32(1)]
PSH Flag Cnt [np.uint32(0), np.uint32(1)]
ACK Flag Cnt [np.uint32(0), np.uint32(1)]
URG Flag Cnt [np.uint32(0), np.uint32(1)]
CWE Flag Count [np.uint32(0), np.uint32(1)]
ECE Flag Cnt [np.uint32(0), np.uint32(1)]
Label ['Benign', 'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP', 'DoS attacks-GoldenEye', 'DoS attacks-Slowloris', 'FTP-BruteForce', 'Infilteration', 'SSH-Bruteforce']
Dst Port Cat ['dfs', 'ephemeral', 'ftp', 'registered', 'ssh', 'web', 'well_known_other']
attack_group ['Benign', 'Bruteforce', 'DDOS', 'DoS', 'Infiltration']
attack_or_benign ['Attack', 'Benign']


#### 2) Codificación de las variables simbólicas

Primero divido el conjunto de variables simbólicas en sus diferentes grupos y los codifico acorde a su tipo.

In [4]:
# Symbolic var config
target_col = "attack_or_benign"   # "Label", "attack_or_benign" or "attack_group"
binary_cols = [
    "Fwd PSH Flags", "Fwd URG Flags",
    "FIN Flag Cnt", "SYN Flag Cnt",
    "RST Flag Cnt", "PSH Flag Cnt",
    "ACK Flag Cnt", "URG Flag Cnt",
    "CWE Flag Count", "ECE Flag Cnt"
]
categorical_cols = ["Protocol", "Dst Port Cat"]

Preparo los conjuntos de variables y respuesta.

In [5]:
# Features and labels
X = dn.drop(columns=['Label', 'attack_group', 'attack_or_benign'])
y = dn[target_col] #I'll start with the binary one which is the easiest
Xn = dn_log.drop(columns=['Label', 'attack_group', 'attack_or_benign'])
yn = dn_log['attack_or_benign'] # Always with the binary one

Ahora hago la codificación:

In [6]:
# Cast binary columns
for col in binary_cols:
    X[col] = X[col].astype("uint8")
    Xn[col] = Xn[col].astype("uint8")

# One-hot encode categoricals
X = pd.get_dummies(X, columns=categorical_cols)
Xn = pd.get_dummies(Xn, columns=categorical_cols)

#### 3) División del conjunto de datos

Ahora procedo a dividir el conjunto en un conjunto de train (80%) y uno de test (20%). Luego validaremos los modelos por validación cruzada por lo que no es necesario hacer un conjunto aparte de validación. Además continuó con la codificación de la variable respuesta.

In [7]:
# df Train (80%) / Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
    shuffle=True
)

# Shapes
print("Dataset dn:")
print("  Train shape:", X_train.shape, y_train.shape)
print("  Test shape:", X_test.shape, y_test.shape)

Dataset dn:
  Train shape: (1688483, 58) (1688483,)
  Test shape: (422121, 58) (422121,)


He creado mi propio objeto para codificar las clases de la variable "Label" y así mantengo la clase "Benign" como la primera, es decir la 0.

In [8]:
le_dn = CustomBenignFirstEncoder()
y_train = le_dn.fit_transform(y_train)
y_test  = le_dn.transform(y_test)
print("Mapping de codificación:", le_dn.get_mapping())

Mapping de codificación: {'Benign': 0, 'Attack': 1}


Ahora hago el **split para los no supervisados**, en este caso uso la transformación logarítmica y luego el estandarizado. Pongo benignos a 0 para no tener problemas luego con los métodos que uso en el modelado:

In [9]:
# 1. Split by class using Pandas indexing (.loc)
X_attack = Xn.loc[yn == 'Attack']
X_benign = Xn.loc[yn == 'Benign']

# 2. TRAIN = ONLY benign (learn normal behavior)
# (train_test_split works perfectly with Pandas DataFrames)
X_train_N, X_benign_test = train_test_split(
    X_benign,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# 3. TEST = unseen benign + attacks
# Equivalent to np.vstack, but uses native Pandas concatenation
X_test_N = pd.concat([X_benign_test, X_attack])

# 4. Generate labels using standard Python lists
y_test_N = [0] * len(X_benign_test) + [1] * len(X_attack)
y_test_N = np.array(y_test_N) # conversion to numpy dataframe

# 5. Shuffle test set 
# (shuffle keeps the DataFrame rows and the label list perfectly aligned)
X_test_N, y_test_N = shuffle(X_test_N, y_test_N, random_state=42)

#### 4) Estandarizado de los datos

Para evitar sesgos hago el estandarizado por separado y guardo los nombres de las columnas para luego ponerselos. Además añado las variables codificadas así tenemos el dataset completo transformado.

In [10]:
scaler_N = RobustScaler().set_output(transform="pandas")

# Fit and scale the entire training set at once
X_train_scaled_N = scaler_N.fit_transform(X_train_N).astype('float32')

# Scale the entire test set using the fitted scaler
X_test_scaled_N = scaler_N.transform(X_test_N).astype('float32')

# Clip the extreme values
#X_train_scaled_N = X_train_scaled_N.clip(lower=-5.0, upper=30.0)
#X_test_scaled_N = X_test_scaled_N.clip(lower=-5.0, upper=30.0)

In [11]:
print("\nFINAL SPLIT:")
print("Train:", X_train_scaled_N.shape)
print("Test:", X_test_scaled_N.shape, y_test_N.shape)


FINAL SPLIT:
Train: (1356273, 58)
Test: (754331, 58) (754331,)


Compruebo los tipos, los cambio para que funcionen mejor con los modelos.

In [12]:
X_train_scaled_N.dtypes

Fwd PSH Flags                    float32
Fwd URG Flags                    float32
FIN Flag Cnt                     float32
SYN Flag Cnt                     float32
RST Flag Cnt                     float32
PSH Flag Cnt                     float32
ACK Flag Cnt                     float32
URG Flag Cnt                     float32
CWE Flag Count                   float32
ECE Flag Cnt                     float32
Down/Up Ratio                    float32
Init Fwd Win Byts                float32
Init Bwd Win Byts                float32
Fwd Seg Size Min                 float32
repetition_count                 float32
repetition_flow                  float32
log_Flow Duration                float32
log_Tot Bwd Pkts                 float32
log_TotLen Fwd Pkts              float32
log_Fwd Pkt Len Max              float32
log_Fwd Pkt Len Min              float32
log_Fwd Pkt Len Mean             float32
log_Fwd Pkt Len Std              float32
log_Bwd Pkt Len Max              float32
log_Bwd Pkt Len 

#### 5) Balanceo del conjunto de train 

A la hora de tratar con el problema de clases desbalanceadas opto por la opción de usar pesos. Para los modelos que he elegido XGBoost y Rand Forest las opciones para balancear vienen practicamente incluidas, sólo tengo que poner la opción.

#### 6) Guardado de los conjuntos de train y test (supervisado y no supervisado)

Guardo los diferentes conjuntos para usarlos en el siguiente notebook de modelado.

**Supervisado:**

In [13]:
pd.DataFrame(y_train).to_parquet(y_train_out, engine="pyarrow", index=False)
pd.DataFrame(y_test).to_parquet(y_test_out, engine="pyarrow", index=False)
pd.DataFrame(X_train).to_parquet(X_train_out, engine="pyarrow", index=False)
pd.DataFrame(X_test).to_parquet(X_test_out, engine="pyarrow", index=False)

**No supervisado:**

In [14]:
pd.DataFrame(y_test_N).to_parquet(y_test_N_out, engine="pyarrow", index=False)
pd.DataFrame(X_train_scaled_N).to_parquet(X_train_N_out, engine="pyarrow", index=False)
pd.DataFrame(X_test_scaled_N).to_parquet(X_test_N_out, engine="pyarrow", index=False)